# Problem 4 - Golden Mountain

For the following problem, let's try to follow the code as suggested in the BlueQubit [tutorial](https://app.bluequbit.io/tutorial/91OgcR5O1GqFpues)

First load the relevant libraries

In [1]:
# --- Setup ---
import os
import time
import warnings

# Optional internal flags you were using
# os.environ["BLUEQUBIT_PPS_DO_NO_USE_PARALLEL_COMPUTE"] = "1"
os.environ["BLUEQUBIT_DEQART_INTERNAL_DISABLE_STRICT_VALIDATIONS"] = "1"

warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
from qiskit import QuantumCircuit, transpile
import bluequbit

# Optional: load .env file if present (pip install python-dotenv)
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

# --- Auth ---
token = os.getenv("BLUEQUBIT_API_TOKEN")
if not token:
    raise RuntimeError(
        "Missing BLUEQUBIT_API_TOKEN. Set it in your shell or in a local .env file."
    )

bq = bluequbit.init(token)
print("BlueQubit client initialized successfully.")

[BQ-PYTHON-SDK][WARNING] - Beta version 0.18.1b1 of BlueQubit Python SDK is being used.


[BQ-PYTHON-SDK][INFO] - There is a newer version of BlueQubit Python SDK available on PyPI. We recommend upgrading. Run 'pip install --upgrade bluequbit' to upgrade from your version 0.18.1b1 to 0.18.3b1.
BlueQubit client initialized successfully.


Now, we can load the corresponding circuit

In [2]:
qc = QuantumCircuit.from_qasm_file('problems/P4_golden_mountain.qasm')

Let's visualize the circuit

In [4]:
print(f"Gates: {qc.count_ops()}")
#qc.draw("mpl", fold=-1)

Gates: OrderedDict([('u3', 10240), ('cz', 5096)])


This circuit is probably complicated since it takes a long time to even draw. We cannot solve it anymore with BlueQubit's Statevector Simulator as given in bluequbit's [tutorial](https://app.bluequbit.io/tutorial/91OgcR5O1GqFpues). The reason for it is that `device='cpu'` won't work anymore in the free version, since it's limited to max. 34 qubits as seen in bluequbit's [doc](https://app.bluequbit.io/docs#devices). Therefore, we switch to `device='mps.cpu'` which can handle up to 96 qubits. This is the MPS Simulator based on Quimb library, on CPUs. Let's try the standard settings in the chapter `Tree Line: MPS Simulations` of BlueQubit's tutorial mentioned before. In order to learn something from it, let's shortly explain what the options mean:
*  Bond dimension (mps_bond_dimension): The bond dimension controls how much entanglement the MPS representation is allowed to capture.
During each gate application, the MPS is compressed using an SVD, and only the $\chi$ largest Schmidt coefficients are retained, where $\chi$ is the chosen bond dimension. A larger bond dimension produces a more faithful approximation of the true quantum state, but at the cost of increased memory usage and longer simulation time.
* Number of shots (shots): For a fixed MPS approximation, the number of shots determines the statistical accuracy of the sampled output distribution. Higher shot counts yield histograms that more closely match the probabilities encoded in the MPS state. However, increasing the number of shots directly increases runtime, since each shot corresponds to an independent sample drawn from the MPS.

## Standard Settings

In [5]:
# Run the circuit using BlueQubit's MPS simulator.

results_mps = bq.run(
    qc,
    device="mps.cpu",
    shots=500,
    options={"mps_bond_dimension": 32},
)

print(f"MPS simulation runtime: {results_mps.run_time_ms} ms")

[BQ-PYTHON-SDK][INFO] - Submitted: Job ID: fHd0xIWlj7mwVM8K, device: mps.cpu, run status: RUNNING, created on: 2026-03-05 20:59:33 UTC, estimated runtime: 156908 ms, estimated cost: $0.00, num qubits: 48, shots: 500
MPS simulation runtime: 44211 ms


In [6]:
# Extract the sampled bitstring counts from the MPS simulation
counts = results_mps.get_counts()

# Sort the bitstrings by frequency (descending) and take the top 10
top10_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:10]

# The bitstring with the highest count is our candidate peak bitstring
peak_bitstring_mps = top10_counts[0][0]

print("Top 10 most frequent bitstrings from MPS Simulation: \n")
for bitstring, count in top10_counts:
    print(f"  {bitstring}: {count}")
print(f"\nPeak bitstring inferred from MPS samples: {peak_bitstring_mps}")

Top 10 most frequent bitstrings from MPS Simulation: 

  111011111011100001000000101101100000011111011010: 1
  100011000111011010110001111100110010110111001011: 1
  001101100000101001101100110111011010101110100011: 1
  100100000111011000000101011101000000111100101101: 1
  110001011100010010111001000110000100100111100110: 1
  011011010110111011001010010011111101100110111101: 1
  110100111010011000111000011111110010100100101011: 1
  000111101000010110010110101111101111110101011100: 1
  101000100011010110100101000011100000110010001101: 1
  110101101110001000100110101100011000111100101100: 1

Peak bitstring inferred from MPS samples: 111011111011100001000000101101100000011111011010


All bitstrings show the same result `1`. So, this approach clearly didn't work out. 

## Increase Settings

Let's see if we can increase the number of shots or bond dimension:

In [10]:
# Run the circuit using BlueQubit's MPS simulator.

results_mps = bq.run(
    qc,
    device="mps.cpu",
    shots=1000,
    options={"mps_bond_dimension": 128},
)

print(f"MPS simulation runtime: {results_mps.run_time_ms} ms")

BQJobNotCompleteError: Job gsuSrYTljBzLpATD finished with status: NOT_ENOUGH_FUNDS. Estimated cost of the job: $0.34. Current balance: $0.00. Current pending jobs estimated cost: $0.00. Minimum balance required: $0.68.

In [8]:
# Extract the sampled bitstring counts from the MPS simulation
counts = results_mps.get_counts()

# Sort the bitstrings by frequency (descending) and take the top 10
top10_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:10]

# The bitstring with the highest count is our candidate peak bitstring
peak_bitstring_mps = top10_counts[0][0]

print("Top 10 most frequent bitstrings from MPS Simulation: \n")
for bitstring, count in top10_counts:
    print(f"  {bitstring}: {count}")
print(f"\nPeak bitstring inferred from MPS samples: {peak_bitstring_mps}")

Top 10 most frequent bitstrings from MPS Simulation: 

  000101100100000111100110101111100110101000010110: 1
  110111001110110100011110001111010010100001000000: 1
  011111001110001001000101001110011111010000110010: 1
  011111010010001110111110011000110011000011000010: 1
  110111110101011101011110001001010010001010110111: 1
  011001010010001000000111000111101010100001111110: 1
  000101000100001000100110000110101100111110011110: 1
  101010101001001101001101000100010101000110001101: 1
  011000011001101001010110101110000111011110001101: 1
  110110011110001000000110101100110010001001000000: 1

Peak bitstring inferred from MPS samples: 000101100100000111100110101111100110101000010110


It looks like increased shots and bond dimension either gave the same results, or were not possible due to `not enough funding`. So we need to switch to another method.